In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
from numpy.linalg import eigvals, det, matrix_rank

In [16]:
df = pd.read_csv("FinalProject.csv")

In [17]:
df.columns = df.columns.str.lower().str.strip()

In [18]:
permno_map = {
    84398: "SPY",
    89469: "IEF",
    90448: "GLD"
}

df["ticker"] = df["permno"].map(permno_map)
df = df[df["ticker"].notna()].copy()

print(df["ticker"].value_counts())

ticker
SPY    1005
IEF    1005
GLD    1005
Name: count, dtype: int64


In [19]:
df = df[(df["prc"].notna()) & (df["cfacpr"].notna()) & (df["cfacpr"] != 0)].copy()
df["adj_close"] = df["prc"].abs() / df["cfacpr"]

prices = (
    df[["date", "ticker", "adj_close"]]
    .drop_duplicates(subset=["date", "ticker"])
    .pivot(index="date", columns="ticker", values="adj_close")
    .sort_index()
)

prices = prices[["SPY", "IEF", "GLD"]]

In [20]:
prices = prices.dropna(how="any").copy()
returns = prices.pct_change().dropna().copy()
returns = returns[["SPY", "IEF", "GLD"]]

# Deliverable No. 2

In [21]:
from pathlib import Path

outdir = Path("project3_outputs")
outdir.mkdir(exist_ok=True)

returns.to_csv(outdir / "clean_return_matrix.csv")

In [22]:
mu_hat = returns.mean()
print(mu_hat)
mu_hat.to_csv(outdir / "mean_vector.csv")

ticker
SPY    0.000516
IEF   -0.000246
GLD    0.000323
dtype: float64


In [23]:
Sigma_hat = returns.cov()
print(Sigma_hat)

Sigma_hat.to_csv(outdir / "covariance_matrix.csv")

ticker       SPY       IEF       GLD
ticker                              
SPY     0.000109  0.000005  0.000015
IEF     0.000005  0.000026  0.000019
GLD     0.000015  0.000019  0.000081


In [24]:
np.save(outdir / "mu_hat.npy", mu_hat.values)
np.save(outdir / "Sigma_hat.npy", Sigma_hat.values)

# Part 3

In [25]:
def simulate_returns(mu, Sigma, T, seed=None):
    rng = np.random.default_rng(seed)
    sim = rng.multivariate_normal(mu, Sigma, size=T)
    return pd.DataFrame(sim, columns=["SPY", "IEF", "GLD"])

In [26]:
sim = simulate_returns(
    mu=mu_hat.values,
    Sigma=Sigma_hat.values,
    T=5000,
    seed=42
)

print(sim.head())

        SPY       IEF       GLD
0 -0.006760 -0.001629  0.007562
1 -0.016734  0.008306  0.008472
2 -0.002040  0.000338  0.002024
3  0.012381 -0.004245 -0.000980
4  0.004672 -0.004924 -0.007768


In [27]:
print("Historical mean:\n", mu_hat)
print("\nSimulated mean:\n", sim.mean())

print("\nHistorical cov:\n", Sigma_hat)
print("\nSimulated cov:\n", sim.cov())

Historical mean:
 ticker
SPY    0.000516
IEF   -0.000246
GLD    0.000323
dtype: float64

Simulated mean:
 SPY    0.000443
IEF   -0.000103
GLD    0.000250
dtype: float64

Historical cov:
 ticker       SPY       IEF       GLD
ticker                              
SPY     0.000109  0.000005  0.000015
IEF     0.000005  0.000026  0.000019
GLD     0.000015  0.000019  0.000081

Simulated cov:
           SPY       IEF       GLD
SPY  0.000109  0.000005  0.000016
IEF  0.000005  0.000027  0.000020
GLD  0.000016  0.000020  0.000082
